# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [100]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [101]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [102]:
fraud["fraud"].value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [103]:
# Select the features (X) and the target (y)
X = fraud.drop(columns=['fraud'])
y = fraud['fraud']
SEED= 1

In [104]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
# set_output(transform="pandas") makes the scaler return a DataFrame (keeps the real column names and index)

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean             26.7017                          5.0398   
std              66.5358                         26.3318   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                          1.8218           0.8816     0.3504   
std                           2.8080           0.3230     0.4771   

      used_pin_number  online_order  
mean           0.1004        0.6508  
std            0.3006        0.4767  

AFTER scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean                 0.0                             0.0   
std                  1.0                             1.0   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                            -0.0              0.0        0.0   
std                              1.0              1.0        1.0   

      used_pin_number  online_order  
mean              0.0   

In [105]:
# Step 2: Train / Test Split
# --------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y)  # Keeps the same malignant/benign proportion in both sets

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (800000, 7)
X_test shape:  (200000, 7)


In [106]:

# loss='log_loss' --> same Sigmoid + Cost Function from the theory!
sgd_model = SGDClassifier(loss='log_loss', random_state=SEED)
sgd_model.fit(X_train_scaled_df, y_train)  # Fit on train

# The learned weights (w) = one per feature
weights = pd.Series(sgd_model.coef_[0], index=X_train_scaled_df.columns)
# In Regression the answer is a single continuous value --> one coefficient vector --> coef_
# In binary Classification the answer is also a single outcome --> one coefficient vector --> coef_[0]
# (already distinguishes between the two classes) 
# In multiclass Classification, each class needs its own weight vectors --> coef_[0], coef_[1], coef_[2]...

# Sorted by |w| so it is easy to read, but keeping the sign to show the direction of the effect
print('Learned weights (Wi):')
print(weights.round(4).reindex(abs(weights).sort_values(ascending=False).index))

Learned weights (Wi):
used_pin_number                  -3.4432
online_order                      2.9391
ratio_to_median_purchase_price    2.4989
distance_from_home                1.0072
distance_from_last_transaction    0.5461
used_chip                        -0.5085
repeat_retailer                  -0.2267
dtype: float64


In [107]:
# Step 5 - Results
# ----------------

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = sgd_model.predict(X_train_scaled_df)     # train
y_pred_test = sgd_model.predict(X_test_scaled_df)       # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['not_fraud', 'fraud']))

Accuracy (Train): 95.94%

              precision    recall  f1-score   support

   not_fraud       0.96      0.99      0.98    182519
       fraud       0.89      0.61      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



In [108]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

ros = RandomOverSampler(random_state=SEED)
X_train_over, y_train_over = ros.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_over.value_counts())

oversampling_model = SGDClassifier(loss='log_loss',random_state=SEED)
oversampling_model.fit(X_train_over, y_train_over)  # Fit on the oversampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
1.0    730078
0.0    730078
Name: count, dtype: int64
Model trained!


In [109]:

# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = oversampling_model.predict(X_train_scaled_df)     # train
y_pred_test = oversampling_model.predict(X_test_scaled_df)       # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['not_fraud', 'fraud']))

Accuracy (Train): 93.38%

              precision    recall  f1-score   support

   not_fraud       0.99      0.93      0.96    182519
       fraud       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [110]:
rus = RandomUnderSampler(random_state=SEED)
X_train_under, y_train_under = rus.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_under.value_counts())

undersampling_model = SGDClassifier(loss='log_loss',random_state=SEED)
undersampling_model.fit(X_train_under, y_train_under)  # Fit on the undersampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
0.0    69922
1.0    69922
Name: count, dtype: int64
Model trained!


In [111]:
# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = undersampling_model.predict(X_train_scaled_df)     # train
y_pred_test = undersampling_model.predict(X_test_scaled_df)       # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['not_fraud', 'fraud']))

Accuracy (Train): 92.88%

              precision    recall  f1-score   support

   not_fraud       1.00      0.92      0.96    182519
       fraud       0.55      0.96      0.70     17481

    accuracy                           0.93    200000
   macro avg       0.77      0.94      0.83    200000
weighted avg       0.96      0.93      0.94    200000



In [112]:


# instead of duplicating existing points, until it matches the size of the majority class
sm = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = sm.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_smote.value_counts())

smote_model = SGDClassifier(loss='log_loss', random_state=SEED)
smote_model.fit(X_train_smote, y_train_smote)  # Fit on the SMOTE-resampled Train set

print("Model trained!")

Before: fraud
0.0    730078
1.0     69922
Name: count, dtype: int64
After:  fraud
1.0    730078
0.0    730078
Name: count, dtype: int64
Model trained!


In [113]:
# Step 5 - Results
# ----------------

y_pred_train = smote_model.predict(X_train_scaled_df)
y_pred_test = smote_model.predict(X_test_scaled_df)

print("Train accuracy:", accuracy_score(y_train, y_pred_train))
print("Test report:")
print(classification_report(y_test, y_pred_test))


Train accuracy: 0.93325625
Test report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.57      0.95      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [114]:
# class_weight='balanced' --> does NOT touch the training data (still 747 vs 53)
# instead, it makes errors on the minority class (Fraud) cost more during training
balanced_model = SGDClassifier(loss='log_loss', class_weight='balanced', random_state=SEED)
balanced_model.fit(X_train_scaled_df, y_train)  # Fit on the ORIGINAL, imbalanced Train set

print("Model trained!")

Model trained!


In [115]:
y_pred_train = balanced_model.predict(X_train_scaled_df)
y_pred_test = balanced_model.predict(X_test_scaled_df)

accuracy_train = accuracy_score(y_train, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 93.38%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182519
       Fraud       0.57      0.94      0.71     17481

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 